In [1]:
# Importando as bibliotecas
import pandas as pd
import pandera.pandas as pa
import pandera.errors as pa_errors
from data_profiling import ProfileReport
from pathlib import Path

ROOT = Path.cwd().parents[0]
print(ROOT)

DATA = ROOT / "data" / "raw"
REPORTS = ROOT / "reports"


c:\Users\yanchagas04\Documents\dev\machine_learning_cimatec\olist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\yanchagas04\Documents\dev\machine_learning_cimatec\olist


In [2]:
# carregando as orders
orders = pd.read_csv(
    DATA / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [3]:
# Criar um profile do dataset
profile = ProfileReport(
    orders,
    title="Data Profiling - Olist Orders",
    minimal=True,
    explorative=False
)

profile.to_file(REPORTS / "profile_olist_orders_profile.html")

# ou abrir por: profile.to_notebook_iframe()
# ou gerar html: profile.to_file("olist_orders_profile.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 61.15it/s]


In [5]:
profile

# Data Quality Dimensions


In [8]:
# ---------------------------------------------------------
# 2. Definição do contrato de qualidade da tabela orders
# ---------------------------------------------------------
orders_schema = pa.DataFrameSchema(
    {

        # -------------------------------------------------
        # UNICIDADE + COMPLETUDE
        # -------------------------------------------------
        # Cada pedido deve possuir um identificador.
        # Como a granularidade desta tabela é:
        #
        # 1 linha = 1 pedido
        #
        # o order_id também deve ser único.
        "order_id": pa.Column(
            str,
            nullable=False,   # não pode ser nulo
            unique=True       # não pode aparecer repetido
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Todo pedido deve estar associado a um cliente.
        "customer_id": pa.Column(
            str,
            nullable=False
        ),


        # -------------------------------------------------
        # VALIDADE
        # -------------------------------------------------
        # O status do pedido deve pertencer ao conjunto
        # de valores esperados no processo da Olist.
        "order_status": pa.Column(
            str,
            pa.Check.isin([
                "created",
                "approved",
                "invoiced",
                "processing",
                "shipped",
                "delivered",
                "unavailable",
                "canceled",
            ]),
            nullable=False
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Precisamos saber quando o pedido foi realizado.
        "order_purchase_timestamp": pa.Column(
            "datetime64[ns]",
            nullable=False
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # A aprovação pode estar ausente em alguns casos,
        # por exemplo dependendo do status do pedido.
        #
        # Portanto, inicialmente permitimos NULL.
        #
        # Depois podemos criar uma regra mais específica:
        # "se o pedido foi aprovado/entregue,
        # então order_approved_at deve existir".
        "order_approved_at": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Nem todo pedido necessariamente chegou à transportadora.
        # Pedidos cancelados, por exemplo, podem não possuir essa data.
        "order_delivered_carrier_date": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Nem todo pedido necessariamente foi entregue.
        # Logo, NULL aqui não significa automaticamente erro.
        "order_delivered_customer_date": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # O prazo prometido é fundamental para nosso problema,
        # porque será usado para determinar se houve atraso.
        "order_estimated_delivery_date": pa.Column(
            "datetime64[ns]",
            nullable=False
        ),
    },


    # -----------------------------------------------------
    # 3. Regras de CONSISTÊNCIA entre colunas
    # -----------------------------------------------------
    checks=[

        # A aprovação do pagamento não deveria ocorrer
        # antes da criação/compra do pedido.
        #
        # Quando order_approved_at é NULL,
        # não consideramos isso uma violação desta regra.
        pa.Check(
            lambda df:
                df["order_approved_at"].isna()
                |
                (
                    df["order_approved_at"]
                    >= df["order_purchase_timestamp"]
                ),
            error=(
                "order_approved_at não pode ocorrer "
                "antes de order_purchase_timestamp"
            )
        ),

    ]
)


# ---------------------------------------------------------
# 4. Validação
# ---------------------------------------------------------
# lazy=True é útil porque o Pandera tenta encontrar
# várias violações antes de gerar o erro.
#
# Sem lazy=True, ele pode parar logo no primeiro problema.


try:
    validated_orders = orders_schema.validate(
        orders,
        lazy=True
    )

    print("Dados válidos!")

except pa_errors.SchemaErrors as err:
    print("Foram encontrados problemas de qualidade.")
    display(err.failure_cases)


Dados válidos!


In [7]:
validated_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
